# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [37]:
# 1. Import libraries and set basic variables

import sys
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from itertools import product, repeat
import geopandas as gpd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import shutil
import json
import yaml
from datetime import datetime
from tqdm import tqdm
import copy

root_path = Path(globals()['_dh'][0]).resolve().parent
sys.path.append(str(root_path))

from paths import project_path, pipeline_path, input_path, api_path, data_path
from library.utilities import get_path, sort_params, reset_parameters, clear_api, get_scenarios, get_filters, is_base_resolution
from library.randomizer import randomize_demand, random_factor_time_series
from library.growth import generate_growth_time_series
import library.scenario_constraints

In [30]:
# 2. Load configuration (new version, come back to this later)

with open(project_path / 'config.yaml', "r") as f:
    conf = yaml.safe_load(f)
    with open(project_path / conf['partitionConfig'], "r") as tf:
        conf['partition'] = yaml.safe_load(tf)
        del conf['partitionConfig']
'''
    with open(project_path / config['pipelineConfig'], "r") as pf:
        config['pipeline'] = yaml.safe_load(pf)
        del config['pipelineConfig']
    with open(project_path / config['apiConfig'], "r") as af:
        config['api'] = yaml.safe_load(af)
        del config['apiConfig']
'''

'\n    with open(project_path / config[\'pipelineConfig\'], "r") as pf:\n        config[\'pipeline\'] = yaml.safe_load(pf)\n        del config[\'pipelineConfig\']\n    with open(project_path / config[\'apiConfig\'], "r") as af:\n        config[\'api\'] = yaml.safe_load(af)\n        del config[\'apiConfig\']\n'

In [3]:
# 2. Load the configuration

with (pipeline_path / 'county-prototype.json').open('r', encoding='utf-8') as file:
    config = json.load(file)

## Set flag to clear the api folder
clear_api_flag = True

In [4]:
# 3. Calculate the scenarios

## Extract and load the constraint functions
constraint_names = [ scenario['name'] for scenario in config['scenarios'] if scenario['type'] == 'constraint' and scenario['disable'] == False ]
constraint_functions = [ getattr(library.scenario_constraints, name) for name in constraint_names ]

# Extract names and values from non-constraint parameters
parameter_objs = [obj for obj in config['scenarios'] if obj['type'] != 'constraint' and obj['disable'] == False]
names = [obj['name'] for obj in parameter_objs]
values = [[item['value'] for item in obj['items']] for obj in parameter_objs]

# Build default scenario
default_scenario = {
    obj["name"]: obj["default"]
    for obj in parameter_objs
    if "default" in obj
}

# Validate default scenario
if not all(fn(default_scenario) for fn in constraint_functions):
    raise ValueError(f"Default scenario violates constraints: {default_scenario}")

# Generate all scenarios
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Filter using constraints and mark the default
scenarios = []
for s in all_scenarios:
    if all(fn(s) for fn in constraint_functions):
        s_out = dict(s)
        if s == default_scenario:
            s_out["default"] = True
        scenarios.append(s_out)

In [5]:
# 3. Load the base demand

base_demand = pd.read_csv(input_path / input_path / config['base']['access'] / f"{config['base']['name']},{sort_params(config['base']['properties'])}.csv", index_col=['timestamp'], parse_dates=['timestamp'])

In [6]:
# 4 Execute transformations - Sort the transformations array

transformers = config['transformers']
transformers.sort(key=lambda x: x["order"])

In [7]:
# 4.1. Execute transformations - Split base demand (national) into geos

transformer = transformers[0]
inputs = transformer['inputs']
input = pd.read_csv(input_path / transformer['access'] / inputs[0]['path'], usecols=inputs[0]['columns'], dtype={inputs[0]['index']: str}, index_col=[inputs[0]['index']])

# Calculate geography demand
geography_demand_matrix = input[input.columns[0]].values[:, None] * base_demand[base_demand.columns[0]].values

# Randomize demand if the transformer has randomness set to true
if transformer['randomness']:
    geography_demand_matrix = randomize_demand(geography_demand_matrix, 0.1)

# Create new dataframe index (includes geography)
geography_index = pd.MultiIndex.from_product([input.index, base_demand.index], names=['geography', 'timestamp'])

# Create a new dataframe on the municipal demand with randomness applied
geography_demand = pd.DataFrame(geography_demand_matrix.ravel(), index=geography_index, columns=['demand'])

In [8]:
# 4.2. Execute transformations - Apply growth over time

## Extract geographies
geographies = geography_demand.index.get_level_values('geography').unique()

## Define growth parameters
start_time = pd.Timestamp(config['start-time'])
end_time = pd.Timestamp(config['end-time'])

new_timestamps = pd.date_range(start=start_time, end=end_time, freq=config['base']['properties']['resolution'])

## Extract hourly demand patterns for 2024
hourly_demand_2024 = (
    geography_demand.loc[
        geography_demand.index.get_level_values('timestamp').year == 2024
    ]
    .reset_index()
)

# Repeat the 2024 demand pattern to match the new timestamps
hourly_demand_pattern = hourly_demand_2024.groupby("geography")["demand"].apply(
    lambda x: np.tile(x.values, len(new_timestamps) // len(x) + 1)[:len(new_timestamps)]
)

# Convert hourly demand pattern back to a 2D array (municipalities x timestamps)
base_demand_repeated = np.vstack(hourly_demand_pattern.values)

## Get the scenarios
transformer = transformers[1]
transformer_scenarios = transformer['scenarios']

## Working array
extended_data = []

for transformer_scenario in transformer_scenarios:
    scenario_name = transformer_scenario['name']
    scenario_index = transformer_scenario['index']
    if transformer_scenario['type'] == 'exp-growth-to-target':
        scenario_target = transformer_scenario['target']
    scenario_randomness = transformer_scenario.get('randomness', 0)  # Default to 0 if not provided

    ## Generate the index and growth factors for the extended demand
    growth_factors = generate_growth_time_series(new_timestamps, config['base']['properties']['resolution'], transformer_scenario)

    # Generate randomness
    random_factors = np.random.uniform(-scenario_randomness, scenario_randomness, size=base_demand_repeated.shape)

    # Apply growth factors
    extended_demand_values = base_demand_repeated * growth_factors * (1+random_factors)

    new_index = pd.MultiIndex.from_product(
        [geographies, new_timestamps, [scenario_index]], names=["geography", "timestamp", scenario_name]
    )

    # Create the extended demand DataFrame
    extended_geography_demand = pd.DataFrame(
        data=extended_demand_values.flatten(),
        index=new_index,
        columns=["demand"]
    )

    # Append to the list
    extended_data.append(extended_geography_demand)

extended_geography_scenario_demand = pd.concat(extended_data)

In [11]:
# 0. Grab your configured baseline year
base_year = 2025  # or pull from your config.yaml

raw = extended_geography_scenario_demand['demand']
years = raw.index.get_level_values('timestamp').year
months = raw.index.get_level_values('timestamp').month

# 1. Compute a single‐year (July of base_year) industry baseline
industry = (
    raw
    .where((months == 7) & (years == base_year))       # only July of base_year
    .groupby([
        raw.index.get_level_values('geography'),
        raw.index.get_level_values('growth')
    ])
    .transform('min')                                  # lowest July of that year
    .mul(0.5)                                          # 50% of that
)

# 2. Compute remainder, buildings, transport
remainder = raw.sub(industry)
buildings = remainder.mul(0.95)
transport = remainder.mul(0.05)

# 3. Stack into long‐form Series
segmented = pd.concat(
    {'industry': industry, 'buildings': buildings, 'transport': transport},
    names=['segment']
)

# 4. Reset index & rename to match schema
extended_geography_scenario_sector_demand = (
    segmented
      .rename('value')
      .reset_index()
      .rename(columns={
          'timestamp': 'period.start',
          'geography': 'dimensions.geography.level1',
          'segment':   'dimensions.segment.level1',
          'growth':    'scenario.growth'
      })
)


In [ ]:
''' OLD CODE
# 4.3. Execute transformations - Split demand into ['industry', 'buildings', 'transport']

## TODO: Make this less naive

## This transform is not realistic. It does the following:
##      1. Assumes a flat industrial demand of 50% of the lowest day in July 2024 in each municipality
##      2. Calculates transport and buildings as respectively 5% and 95% of the remainder 

## Extract July data for each year
july_data = extended_geography_scenario_demand.loc[
    extended_geography_scenario_demand.index.get_level_values('timestamp').month == 7
]

# Compute the lowest July value per municipality, year, and growth scenario
lowest_july_per_year = (
    july_data
    .groupby([
        july_data.index.get_level_values('geography'),
        july_data.index.get_level_values('timestamp').year,
        july_data.index.get_level_values('growth')  # Ensure growth scenario is part of grouping
    ])['demand']
    .min()
)

# Convert to DataFrame and calculate industry demand
lowest_july_per_year = lowest_july_per_year.to_frame(name='lowest_july')
lowest_july_per_year['industry_demand'] = lowest_july_per_year['lowest_july'] * 0.5

# Create extended_geography_sector_demand and merge industry demand back with the full extended demand
extended_geography_scenario_sector_demand = extended_geography_scenario_demand.copy()

# Extract year from timestamp in the index
extended_geography_scenario_sector_demand['year'] = extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year

# Join the calculated industry demand on ['geography', 'year', 'growth']
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.join(
    lowest_july_per_year['industry_demand'],
    on=['geography', 'year', 'growth']
)

# Industry demand is constant across each municipality and year
extended_geography_scenario_sector_demand['industry'] = extended_geography_scenario_sector_demand['industry_demand']

# Compute the remainder for buildings and transport
extended_geography_scenario_sector_demand['remainder'] = (
    extended_geography_scenario_sector_demand['demand'] - extended_geography_scenario_sector_demand['industry']
)

# Split remainder into buildings (95%) and transport (5%)
extended_geography_scenario_sector_demand['buildings'] = extended_geography_scenario_sector_demand['remainder'] * 0.95
extended_geography_scenario_sector_demand['transport'] = extended_geography_scenario_sector_demand['remainder'] * 0.05

# Drop unnecessary intermediate columns
extended_geography_scenario_sector_demand = extended_geography_scenario_sector_demand.drop(columns=['demand', 'industry_demand', 'remainder', 'year'])'
'''

**The code below all pertains to writing data.**

In [ ]:
partition_keys = conf['partition']["partitionKeys"]
row_group_size = conf['partition'].get("rowGroupSize", None)

# 1) Convert your pandas DF into an Arrow Table
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

# TODO: Rewrite a more elegant function for transforming the partitioning.yaml (maybe change the format to a schema)
# 2) Compute each derived key in Arrow and append as a real column
#    We’ll also build a list of the *actual* column names to partition on:
actual_partitions = []
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]            # e.g. "period.start"
        tf  = entry["transform"]       # e.g. "year"
        # Compute via pyarrow.compute.<tf>(...)
        arr = getattr(pc, tf)(table[src])
        # Append that as a column named exactly tf
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    else:
        # plain existing column
        actual_partitions.append(entry['path'])

# 3) Now write a Hive‐style dataset over those *real* columns
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,         # list[str] works with hive flavor
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)


In [ ]:
## Clear the api folder
if clear_api_flag:
    print("Clearing API folder...")
    clear_api(data_path)

Clearing API folder...


In [10]:
## Prepare data
print('Preparing data...')

geos = extended_geography_scenario_sector_demand.index.get_level_values('geography').unique()
years = extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year.unique()
growth_scenarios = extended_geography_scenario_sector_demand.index.get_level_values('growth').unique()

base_resolution = config['base']['properties']['resolution']
base_aggregation = config['base']['properties']['aggregation']
resolutions = [obj['value'] for obj in config['filters']['resolution']['options'] if obj['value'] != config['base']['properties']['resolution']]
aggregations = [obj['value'] for obj in config['filters']['aggregation']['options'] if obj['value']]

melted_aggregations = [{"resolution": base_resolution, "aggregation": base_aggregation}] + [{"resolution": r, "aggregation": a} for r, a in product(resolutions, aggregations)]

parameters = {
    'filter': get_filters(config),
    'scenario': get_scenarios(config)
}

Preparing data...


In [ ]:
## Write demand summed over geos, per resolution, aggregation, year, and growth scenario

print("Summing data over geographies...")
parameters['filter']['sector'] = 'all'
parameters['filter']['geography'] = '00'
country_data = extended_geography_scenario_sector_demand.groupby(['timestamp', 'growth']).sum()

for year in tqdm(years, desc="Writing total geography data per year", unit="years"):
    parameters['filter']['year'] = year
    yearly_data = country_data.loc[country_data.index.get_level_values('timestamp').year == year]
    for growth_scenario in growth_scenarios:
        # Filter data for the current growth scenario
        parameters['scenario']['growth'] = growth_scenario
        scenario_data = yearly_data.xs(growth_scenario, level='growth')
        for agg in melted_aggregations:
            parameters['filter']['resolution'] = agg['resolution']
            parameters['filter']['aggregation'] = agg['aggregation']
            (scenario_data.resample(agg['resolution']).agg(agg['aggregation']) if not is_base_resolution(agg['resolution'],config) else scenario_data).to_csv(
                data_path / f"demand-t,{sort_params(parameters,['filter','scenario'])}.csv")

parameters = reset_parameters(parameters)

Summing data over geographies...


Writing total geography data per year: 100%|██████████| 20/20 [00:11<00:00,  1.79years/s]


In [12]:
## Group data by year and geography for all aggregations

print('Aggregating data per year...')
grouped = extended_geography_scenario_sector_demand.groupby([extended_geography_scenario_sector_demand.index.get_level_values('timestamp').year, 'geography', 'growth'])
yearly_stats = {stat: grouped.agg(stat) for stat in aggregations}

Aggregating data per year...


In [ ]:
## Write all years per geography

parameters['filter']['sector'] = 'all'
parameters['filter']['resolution'] = '1YE'
parameters['filter']['year'] = 'all'

for geo in tqdm(geos, desc="Progress (geos)", unit="geo"):
    parameters['filter']['geography'] = geo
    for agg in aggregations:
        parameters['filter']['aggregation'] = agg
        for growth_scenario in growth_scenarios:
            parameters['scenario']['growth'] = growth_scenario
            geo_data = yearly_stats[agg].xs((geo, growth_scenario), level=['geography', 'growth'])
            geo_data.to_csv(data_path / f"demand,{sort_params(parameters, ['filter', 'scenario'])}.csv.gz", compression='gzip')

parameters = reset_parameters(parameters)

Progress (geos): 100%|██████████| 21/21 [00:00<00:00, 110.55geo/s]


In [ ]:
## Write total geography per year and growth scenario

parameters['filter']['geography'] = 'all'
parameters['filter']['sector'] = 'all'
parameters['filter']['resolution'] = '1YE'

for year in tqdm(years, desc="Progress (years)", unit="years"):
    parameters['filter']['year'] = year	
    for agg in aggregations:
        parameters['filter']['aggregation'] = agg
        for growth_scenario in growth_scenarios:
            parameters['scenario']['growth'] = growth_scenario
            year_data = yearly_stats[agg].loc[year]
            year_data_scenario = year_data.xs(growth_scenario, level='growth').copy()
            year_data_scenario.loc['00'] = year_data_scenario.agg(agg)
            year_data_scenario.to_csv(data_path / f"demand,{sort_params(parameters, ['filter', 'scenario'])}.csv.gz", compression='gzip')

parameters = reset_parameters(parameters)

Progress (years): 100%|██████████| 20/20 [00:00<00:00, 64.14years/s]


In [ ]:
## Write all years for the total geography, per growth scenario

parameters['filter']['geography'] = '00'
parameters['filter']['sector'] = 'all'
parameters['filter']['resolution'] = '1YE'
parameters['filter']['year'] = 'all'

for agg in aggregations:
    parameters['filter']['aggregation'] = agg
    for growth_scenario in growth_scenarios:
        parameters['scenario']['growth'] = growth_scenario
        national_data = yearly_stats['sum'].xs(growth_scenario, level='growth').groupby(level='timestamp').agg(agg)
        national_data.to_csv(data_path / f"demand,{sort_params(parameters, ['filter', 'scenario'])}.csv.gz", compression='gzip')

parameters = reset_parameters(parameters)

In [ ]:
## Write a single file for all geographies, years, and growth scenarios, per aggregation

parameters['filter']['geography'] = 'all'
parameters['filter']['sector'] = 'all'
parameters['filter']['resolution'] = '1YE'
parameters['filter']['year'] = 'all'
parameters['scenario']['growth'] = 'all'
parameters['filter']['aggregation'] = 'sum'

year_data = yearly_stats[parameters['filter']['aggregation']]
pd.concat([year_data, year_data.groupby(level=['timestamp', 'growth']).agg(parameters['filter']['aggregation']).assign(geography='00').set_index('geography', append=True).reorder_levels(['timestamp', 'geography', 'growth']).sort_index()]).sort_index().to_csv(
    data_path / f"demand,{sort_params(parameters, ['filter', 'scenario'])}.csv.gz", compression='gzip')

parameters = reset_parameters(parameters)

In [ ]:
# Write globals

globals ={
    'upper_bound': year_data['total'].values.max(),
    'lower_bound': year_data['total'].values.min()
}

(data_path / "globals.json").write_text(json.dumps(globals, indent=4, ensure_ascii=False), encoding='utf-8')

79

In [ ]:
## Write demand per geo and year

parameters['filter']['sector'] = 'all'

def process_geo_year_growth(task, param):
    param = copy.deepcopy(param)
    geo, year, growth_scenario = task  # Unpack the task tuple
    param['filter']['geography'] = geo
    param['filter']['year'] = year
    param['scenario']['growth'] = growth_scenario
    geography_data = extended_geography_scenario_sector_demand.xs(geo, level='geography')
    yearly_data = geography_data.loc[geography_data.index.get_level_values('timestamp').year == year]
    scenario_data = yearly_data.xs(growth_scenario, level='growth')
    
    for agg in melted_aggregations:
        param['filter']['resolution'] = agg['resolution']
        param['filter']['aggregation'] = agg['aggregation']
        (scenario_data.resample(agg['resolution']).agg(agg['aggregation']) if not is_base_resolution(agg['resolution'],config) else scenario_data).to_csv(
            data_path / f"demand-t,{sort_params(param,['filter','scenario'])}.csv.gz",
            compression='gzip'
        )

# Create tasks as (geo, year) pairs
tasks = list(product(geos, years, growth_scenarios))

# Use ProcessPoolExecutor with a top-level function
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(process_geo_year_growth, tasks, repeat(parameters)), total=len(tasks), desc="Geo-Year Pairs"))

parameters = reset_parameters(parameters)

Geo-Year Pairs: 100%|██████████| 1260/1260 [01:14<00:00, 16.96it/s]


In [ ]:
# Write the geojson file (no energy data in it right now. Just a renaming.)

parameters['filter']['division'] = config['geography']['division']

print(f"Writing geo,division={parameters['filter']['division']}.geojson")

shutil.copy(input_path / config['access'] / config['geography']['source'], data_path / f"geo,{sort_params(parameters, ['filter', 'scenario'])}.geojson")

parameters = reset_parameters(parameters)

Writing geo,division=county.geojson


In [ ]:
# Write geo,division=[].json

parameters['filter']['division'] = config['geography']['division']

print(f"Writing geo,division={parameters['filter']['division']}.json")

geographies_gdp = gpd.read_file(input_path / config['access'] / config['geography']['source'], encoding='utf-8')
geographies = geographies_gdp[['geo_id', 'geo_name', 'geo_type']].copy()

## Add Sweden as a whole
new_row = pd.DataFrame({
    "geo_type": ["country"],
    "geo_id": ["00"],
    "geo_name": ["Sverige"]
})

geographies = pd.concat([geographies, new_row], ignore_index=True).sort_values(by='geo_id').reset_index(drop=True)

geographies.to_json(data_path / f"geo,{sort_params(parameters, ['filter', 'scenario'])}.json", orient="records", indent=4, force_ascii=False)

parameters = reset_parameters(parameters)

Writing geo,division=county.json


In [ ]:
# Write aggregations.json

print("Writing aggregations.json")

(data_path / "aggregations.json").write_text(json.dumps(melted_aggregations, indent=4, ensure_ascii=False), encoding='utf-8')


Writing aggregations.json


500

In [ ]:
# Write parameters.json

print("Writing parameters.json")

parameters = {
    'filter': {
        'year': list(range(pd.Timestamp(config['start-time']).year, pd.Timestamp(config['end-time']).year)),
        'division': config['geography']['division'],
        'geography': list(geographies['geo_id']),
        'aggregation': aggregations,
        'sector': [sector['value'] for sector in config['filters']['sector']['options']],
    },
    'scenario': {scenario['name']: [i['value'] for i in scenario['items']] for scenario in [s for s in config['scenarios'] if s['type'] != 'constraint' and s['disable'] == False]}
}

(data_path / "parameters.json").write_text(json.dumps(parameters, indent=4, ensure_ascii=False), encoding='utf-8')


Writing parameters.json


1149

In [ ]:
# Write scenarios.json

(data_path / "scenarios.json").write_text(json.dumps(scenarios, indent=4, ensure_ascii=False), encoding='utf-8')

126

In [ ]:
# Write config.json

(data_path / "config.json").write_text(json.dumps(config, indent=4, ensure_ascii=False), encoding='utf-8')

9931